# Optimasi SVR Menggunakan PSO untuk Peramalan Inflasi Indonesia
Final Project Rekayasa Komputasional — jalankan sel secara berurutan di Google Colab.

**Langkah:** upload `inflasi.csv` ke Colab (ikon folder di kiri), lalu jalankan semua sel.

In [ ]:
!pip install pyswarms -q

In [ ]:
# ===== 1. PREPROCESSING =====
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

np.random.seed(42)
N_LAGS = 12

df = pd.read_csv('inflasi.csv')
df['periode'] = pd.to_datetime(df['periode'])
df = df.sort_values('periode').reset_index(drop=True)

# EDA singkat
plt.figure(figsize=(12, 4))
plt.plot(df['periode'], df['inflasi'], color='steelblue')
plt.title('Inflasi Indonesia (Year-on-Year, %)')
plt.grid(alpha=0.3); plt.show()

values = df['inflasi'].values.reshape(-1, 1)
scaler = MinMaxScaler()
scaled = scaler.fit_transform(values).flatten()

X, y = [], []
for i in range(N_LAGS, len(scaled)):
    X.append(scaled[i-N_LAGS:i]); y.append(scaled[i])
X, y = np.array(X), np.array(y)

split = int(len(X) * 0.8)  # split kronologis, tanpa shuffle
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# ===== 2. BASELINE SVR (SEBELUM OPTIMASI) =====
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error

def evaluasi(y_true, y_pred, label):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f'{label} -> RMSE: {rmse:.4f} | MAE: {mae:.4f} | MAPE: {mape:.2f}%')
    return rmse, mae, mape

svr_base = SVR(kernel='rbf')  # default: C=1.0, gamma='scale', epsilon=0.1
svr_base.fit(X_train, y_train)
pred_base = svr_base.predict(X_test)

y_test_asli = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
pred_base_asli = scaler.inverse_transform(pred_base.reshape(-1, 1)).flatten()
hasil_base = evaluasi(y_test_asli, pred_base_asli, 'SVR Baseline')

In [ ]:
# ===== 3. OPTIMASI PSO =====
import pyswarms as ps
from sklearn.model_selection import TimeSeriesSplit

LB = np.array([0.1, 0.0001, 0.0001])  # [C, gamma, epsilon]
UB = np.array([1000., 10., 0.5])

def fitness(particles):
    scores = []
    tscv = TimeSeriesSplit(n_splits=3)
    for p in particles:
        C, gamma, eps = p
        rmses = []
        for tr, va in tscv.split(X_train):
            m = SVR(kernel='rbf', C=C, gamma=gamma, epsilon=eps)
            m.fit(X_train[tr], y_train[tr])
            rmses.append(np.sqrt(mean_squared_error(y_train[va], m.predict(X_train[va]))))
        scores.append(np.mean(rmses))
    return np.array(scores)

options = {'c1': 1.5, 'c2': 1.5, 'w': 0.7}
optimizer = ps.single.GlobalBestPSO(n_particles=20, dimensions=3,
                                    options=options, bounds=(LB, UB))
best_cost, best_pos = optimizer.optimize(fitness, iters=30)
C_opt, gamma_opt, eps_opt = best_pos
print(f'Optimal: C={C_opt:.4f}, gamma={gamma_opt:.4f}, epsilon={eps_opt:.4f}')

# Kurva konvergensi (bukti optimasi untuk laporan)
plt.figure(figsize=(8, 4))
plt.plot(optimizer.cost_history)
plt.title('Kurva Konvergensi PSO'); plt.xlabel('Iterasi'); plt.ylabel('Fitness (RMSE CV)')
plt.grid(alpha=0.3); plt.show()

In [ ]:
# ===== 4. EVALUASI & PERBANDINGAN =====
svr_pso = SVR(kernel='rbf', C=C_opt, gamma=gamma_opt, epsilon=eps_opt)
svr_pso.fit(X_train, y_train)
pred_pso = svr_pso.predict(X_test)
pred_pso_asli = scaler.inverse_transform(pred_pso.reshape(-1, 1)).flatten()
hasil_pso = evaluasi(y_test_asli, pred_pso_asli, 'SVR-PSO')

# Tabel perbandingan
tabel = pd.DataFrame([
    {'model': 'SVR Baseline', 'RMSE': hasil_base[0], 'MAE': hasil_base[1], 'MAPE': hasil_base[2]},
    {'model': 'SVR-PSO', 'RMSE': hasil_pso[0], 'MAE': hasil_pso[1], 'MAPE': hasil_pso[2]},
])
for m in ['RMSE', 'MAE', 'MAPE']:
    tabel[f'Perbaikan {m} (%)'] = ((tabel[m].iloc[0] - tabel[m]) / tabel[m].iloc[0] * 100).round(2)
print(tabel.to_string(index=False))

# Grafik prediksi aktual vs kedua model
tanggal = df['periode'].iloc[N_LAGS + split:]
plt.figure(figsize=(12, 4))
plt.plot(tanggal, y_test_asli, label='Aktual', color='steelblue')
plt.plot(tanggal, pred_base_asli, label='SVR Baseline', color='gray', ls=':')
plt.plot(tanggal, pred_pso_asli, label='SVR-PSO', color='tomato', ls='--')
plt.title('Perbandingan Prediksi: Sebelum vs Sesudah Optimasi')
plt.legend(); plt.grid(alpha=0.3); plt.show()

# Grafik batang perbandingan metrik
tabel.set_index('model')[['RMSE', 'MAE', 'MAPE']].plot(kind='bar', figsize=(8, 4), rot=0)
plt.title('Perbandingan Kinerja Sebelum vs Sesudah Optimasi')
plt.tight_layout(); plt.show()